## Section 4 — Pipeline de Pretraitement

### Etapes du pipeline

| Etape | Technique | Justification |
|-------|-----------|---------------|
| 1 | Conversion gris | Les radios n'ont pas besoin de couleur |
| 2 | CLAHE | Ameliore le contraste local |
| 3 | Resize 224x224 | Taille standard CNN |
| 4 | Normalisation ImageNet | Obligatoire pour Transfer Learning |
| 5 | Data Augmentation | Reduit overfitting (train uniquement) |

### Strategie de reequilibrage
**Plus de WeightedRandomSampler.** Le reequilibrage est entierement assure
par le GAN (Section 5) qui genere de nouvelles images NORMAL synthetiques.
Le dataset final sera quasi-equilibre (3841 NORMAL vs 3875 PNEUMONIA).

In [ ]:
# ============================================================
# SECTION 4 : PIPELINE DE PRETRAITEMENT
# ============================================================

fixer_seed(42)

# ── Elargissement du val set ──────────────────────────────────
# Le val set original (16 images) est trop petit.
# On copie 10% du test set vers le val set pour stabiliser
# les courbes d'apprentissage et l'Early Stopping.

print("Elargissement du val set...")
for classe in ["NORMAL", "PNEUMONIA"]:
    test_dir = os.path.join(DATA_DIR, "test", classe)
    val_dir  = os.path.join(DATA_DIR, "val",  classe)
    images   = os.listdir(test_dir)
    for img in random.sample(images, int(len(images)*0.10)):
        shutil.copy(os.path.join(test_dir, img),
                    os.path.join(val_dir,  img))

print("Val set apres elargissement :")
for split in ["val", "test"]:
    for classe in ["NORMAL", "PNEUMONIA"]:
        nb = len(os.listdir(f"{DATA_DIR}/{split}/{classe}"))
        print(f"  {split} | {classe} : {nb} images")

# ── Fonction CLAHE ────────────────────────────────────────────
def appliquer_clahe(image_path):
    """
    Applique le pipeline complet sur une image :
    Gris → CLAHE → Resize 224x224 → Normalisation [0,1]

    Parametres :
    - image_path : chemin vers le fichier image

    Retourne :
    - img : array numpy float64 (224, 224), valeurs [0, 1]
    """
    # Charger en gris (1 canal — radios n'ont pas de couleur pertinente)
    img   = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    # CLAHE : clipLimit limite le bruit, tileGridSize = taille des zones locales
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img   = clahe.apply(img)
    # Resize vers la taille standard des CNN modernes
    img   = cv2.resize(img, (224, 224))
    # Normalisation [0,255] → [0,1]
    return img / 255.0

# ── Transformations ───────────────────────────────────────────

# Pour le TRAIN : avec augmentation aleatoire
# Chaque image est vue differemment a chaque epoch
transform_train = transforms.Compose([
    transforms.ToPILImage(),
    # Flip horizontal : les poumons sont symetriques gauche/droite
    transforms.RandomHorizontalFlip(p=0.5),
    # Rotation : les radios ne sont pas toujours parfaitement droites
    transforms.RandomRotation(degrees=10),
    # Translation : variabilite de positionnement du patient
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    # Luminosite : variabilite d'exposition radiologique
    transforms.ColorJitter(brightness=0.2),
    # Zoom : variabilite de la distance patient/appareil
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.ToTensor(),
    # Normalisation ImageNet : OBLIGATOIRE pour le Transfer Learning
    # ResNet, DenseNet, EfficientNet ont ete pre-entraines avec ces stats
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Pour VAL et TEST : sans augmentation
# On veut evaluer sur des images realistes, pas transformees
transform_val = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── Dataset personnalise ──────────────────────────────────────
class PneumoniaDataset(Dataset):
    """
    Dataset personnalise pour les radiographies pulmonaires.

    Labels :
    - 0 = NORMAL
    - 1 = PNEUMONIA

    Parametres :
    - data_dir  : chemin vers le dossier (train, val ou test)
    - train     : True = augmentation activee
    - avec_gan  : True = inclut les images synthetiques GAN
    """
    def __init__(self, data_dir, train=False, avec_gan=False):
        self.images    = []
        self.labels    = []
        self.transform = transform_train if train else transform_val

        # Charger NORMAL et PNEUMONIA originaux
        for label, classe in enumerate(["NORMAL", "PNEUMONIA"]):
            dossier = os.path.join(data_dir, classe)
            for f in os.listdir(dossier):
                if f.endswith((".jpg", ".jpeg", ".png")):
                    self.images.append(os.path.join(dossier, f))
                    self.labels.append(label)

        # Ajouter les images GAN si disponibles et demandees
        # Ces images sont des NORMAL synthetiques generees par le GAN
        if avec_gan:
            gan_dir = os.path.join(data_dir, "GAN_NORMAL")
            if os.path.exists(gan_dir):
                for f in os.listdir(gan_dir):
                    if f.endswith(".png"):
                        self.images.append(os.path.join(gan_dir, f))
                        self.labels.append(0)

        nb_n = self.labels.count(0)
        nb_p = self.labels.count(1)
        print(f"Dataset charge : {len(self.images)} images | "
              f"NORMAL={nb_n} | PNEUMONIA={nb_p} | "
              f"Ratio={nb_n/nb_p:.2f}")

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        # 1. Charger et appliquer CLAHE
        img = appliquer_clahe(self.images[idx])
        # 2. Convertir float [0,1] -> uint8 [0,255] (requis par PIL)
        img = (img * 255).astype(np.uint8)
        # 3. Repeter le canal gris 3 fois -> RGB simule
        #    ResNet/DenseNet/EfficientNet attendent 3 canaux
        img = np.stack([img, img, img], axis=-1)
        # 4. Appliquer transformations
        img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

# ── DataLoaders ───────────────────────────────────────────────
def get_dataloaders(data_dir, batch_size=32, avec_gan=False):
    """
    Cree les DataLoaders train/val/test.

    Note importante : Plus de WeightedRandomSampler !
    Le reequilibrage est assure par les images GAN dans le dataset.
    On utilise simplement shuffle=True pour melanger les donnees.
    """
    train_ds = PneumoniaDataset(
        os.path.join(data_dir, "train"), train=True, avec_gan=avec_gan)
    val_ds   = PneumoniaDataset(os.path.join(data_dir, "val"),  train=False)
    test_ds  = PneumoniaDataset(os.path.join(data_dir, "test"), train=False)

    # shuffle=True pour le train : melange les donnees a chaque epoch
    # shuffle=False pour val/test : evaluation reproductible
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    print(f"\nTrain  : {len(train_ds)} images | {len(train_loader)} batchs")
    print(f"Val    : {len(val_ds)} images | {len(val_loader)} batchs")
    print(f"Test   : {len(test_ds)} images | {len(test_loader)} batchs")

    return train_loader, val_loader, test_loader

# ── Visualisation Data Augmentation ──────────────────────────
# Montrer que la meme image est vue differemment a chaque fois
# C'est ce qui permet de reduire l'overfitting

dataset_aug = PneumoniaDataset(os.path.join(DATA_DIR, "train"), train=True)
idx_normal  = dataset_aug.labels.index(0)
idx_pneumo  = dataset_aug.labels.index(1)

fig, axes = plt.subplots(2, 8, figsize=(22, 6))
fig.suptitle("Data Augmentation — Meme image vue 8 fois differemment",
             fontsize=12, fontweight="bold")

for i, (idx, couleur, classe) in enumerate([
    (idx_normal, "steelblue", "NORMAL"),
    (idx_pneumo, "tomato",    "PNEUMONIA")
]):
    for j in range(8):
        img, _ = dataset_aug[idx]
        # Denormaliser pour affichage
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        img_d = (img * std + mean).clamp(0,1)[0].numpy()
        axes[i][j].imshow(img_d, cmap="gray")
        axes[i][j].axis("off")
        if j == 0:
            axes[i][j].set_title(f"{classe}\nV{j+1}", color=couleur, fontsize=8)
        else:
            axes[i][j].set_title(f"V{j+1}", fontsize=8)

plt.tight_layout(); plt.show()
print("Observation : chaque version presente une rotation, luminosite ou cadrage different.")
print("Le modele apprend a reconnaitre la pneumonie quelle que soit la presentation.")
print("\nPipeline pret !")